In [ ]:
import pandas as pd

raw_detailed_df = pd.read_csv("./ADSP_PHC_PET_Amyloid_Detailed_22Aug2025.csv")
raw_simple_df = pd.read_csv("./ADSP_PHC_PET_Amyloid_Simple_22Aug2025.csv")
raw_df = pd.merge(raw_detailed_df, raw_simple_df, on=["RID", "PHC_Age_PET"], how="outer")
raw_df = raw_df.loc[:, ~raw_df.columns.str.endswith("_y")]
raw_df = raw_df.rename(columns=lambda x: x[:-2] if x.endswith('_x') else x)
mapping_df = pd.read_csv("./column_mapping.csv")

print(f"Initial data shape: {raw_df.shape}")

In [ ]:
filters = {
    "PHC_Race": [5],  # 5 = White
    "PHC_Ethnicity": [2],  #  2 = Not Hispanic or Latino
    "PHC_TRACER": ["FBP"]
}

# ---- Apply row filters ----
filtered_df = raw_df.copy()
for col, allowed_values in filters.items():
    if col in filtered_df.columns:
        filtered_df = filtered_df[filtered_df[col].isin(allowed_values)]
        
print(f"Data shape after filtering: {filtered_df.shape}")

In [ ]:
# ---- Column mapping ----
mapping_df = mapping_df[mapping_df["TargetName"].notna() & (mapping_df["TargetName"] != "")]
column_mapping = dict(zip(mapping_df["SourceName"], mapping_df["TargetName"]))

# Ensure all expected columns exist in filtered_df
for src_col in column_mapping.keys():
    if src_col not in filtered_df.columns:
        filtered_df[src_col] = None  # or np.nan if preferred

# Apply mapping
filtered_df = filtered_df[list(column_mapping.keys())].rename(columns=column_mapping)

print(f"Data shape after column mapping: {filtered_df.shape}")

In [ ]:
value_mappings = {
    "DXGrp": {
        1: 1,
        2: 2,
        3: 4
    },
}

# ---- Apply value mappings ----
for col, mapping in value_mappings.items():
    if col in filtered_df.columns:
        filtered_df[col] = filtered_df[col].map(mapping).fillna(-1).astype("Int64")

In [ ]:
# ---- Merge cognitive scores ----
raw_cogn_df = pd.read_csv("./ADSP_PHC_COGN_22Aug2025.csv")

# Step 1: Sort both dataframes to speed up lookups
filtered_df = filtered_df.sort_values(['RID', 'AGE'])
raw_cogn_df = raw_cogn_df.sort_values(['RID', 'PHC_Age_Cognition'])

# Step 2: Define a helper to find the nearest age match for each RID
def match_nearest_age(row):
    rid = row['RID']
    age = row['AGE']
    
    subset = raw_cogn_df[raw_cogn_df['RID'] == rid]
    if subset.empty:
        return pd.Series({'PHC_MEM': None, 'PHC_EXF': None})
    
    # Find index of closest age
    idx = (subset['PHC_Age_Cognition'] - age).abs().idxmin()
    matched = subset.loc[idx]
    return pd.Series({'PHC_MEM': matched['PHC_MEM'], 'PHC_EXF': matched['PHC_EXF']})

# Step 3: Apply to all rows
filtered_df[['PHC_MEM', 'PHC_EXF']] = filtered_df.apply(match_nearest_age, axis=1)

In [ ]:
# ---- Remove data points with missing values ----
# Identify the CTX_ columns
ctx_cols = [col for col in filtered_df.columns if col.startswith("CTX_")]

# Drop rows with NaN in those columns
filtered_df = filtered_df.dropna(subset=ctx_cols)

print(f"Final dataset shape: {filtered_df.shape}")

In [ ]:
# ---- split into single_visit and multi_visit ----
visit_counts = filtered_df['RID'].value_counts()
single_visit_rids = visit_counts[visit_counts == 1].index
multi_visit_rids = visit_counts[visit_counts > 1].index
single_visit_df = filtered_df[filtered_df['RID'].isin(single_visit_rids)]
multi_visit_df = filtered_df[filtered_df['RID'].isin(multi_visit_rids)]

print(f"Single-visit data shape: {single_visit_df.shape}")
print(f"Multi-visit data shape: {multi_visit_df.shape}")

In [ ]:
# Save output
single_visit_df.to_csv("../filtered/single_visit_data.csv", index=False)
multi_visit_df.to_csv("../filtered/multi_visit_data.csv", index=False)